# Просмотр `.mat` — кадры по слайдеру

Выбери файл в списке и двигай **ползунок** (или колёсико на слайдере), чтобы листать кадры.

Kernel: `/Users/user/Education/CVYandexCamp/venv/bin/python`

In [ ]:
DATA_DIR = None               # None → <repo>/data ; или абсолютный путь
PATTERN = "*.mat"             # R_*.mat | Z_*.mat | sample*.mat
SAMPLE_TIME_AXIS = 2          # только для sample*.mat (если авто не угадает)
CMAP = "inferno"              # colormap для тепловизора

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display


def find_project_root() -> Path:
    """Корень репо: есть папки data/ и irt_data/. Не зависит от cwd."""
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    candidates.append(start / "thermal-control-ya-project")
    seen: set[Path] = set()
    for base in candidates:
        for p in [base, *base.parents]:
            p = p.resolve()
            if p in seen:
                continue
            seen.add(p)
            if (p / "data").is_dir() and (p / "irt_data").is_dir():
                return p
    raise FileNotFoundError(
        "Не найден корень репозитория (нужны data/ и irt_data/). "
        f"cwd={start}. Укажи DATA_DIR абсолютным путём."
    )


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from irt_data.cache import _load_mat_array

if DATA_DIR is None:
    data_dir = ROOT / "data"
elif Path(DATA_DIR).is_absolute():
    data_dir = Path(DATA_DIR)
else:
    data_dir = (ROOT / DATA_DIR).resolve()

mat_files = sorted(data_dir.glob(PATTERN))
if not mat_files:
    raise FileNotFoundError(f"Нет файлов {PATTERN} в {data_dir}")

print(f"ROOT={ROOT}")
print(f"Найдено {len(mat_files)} файлов в {data_dir}")

In [ ]:
def _time_axis_for(path: Path) -> int | None:
    return SAMPLE_TIME_AXIS if path.name.lower().startswith("sample") else None


def load_video(path: Path) -> tuple[np.ndarray, dict]:
    video, meta = _load_mat_array(path, time_axis=_time_axis_for(path))
    return video, meta


state: dict = {"video": None, "meta": None, "path": None}

fig, ax = plt.subplots(figsize=(7, 5))
fig.tight_layout()
im = ax.imshow(np.zeros((2, 2)), cmap=CMAP, origin="upper")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
title = ax.set_title("")
ax.axis("off")


def _render_frame(t: int) -> None:
    video = state["video"]
    meta = state["meta"]
    path = state["path"]
    if video is None:
        return
    t = int(np.clip(t, 0, len(video) - 1))
    frame = video[t]
    im.set_data(frame)
    im.set_clim(float(frame.min()), float(frame.max()))
    fps = meta.get("fps")
    time_s = f" | t={t / fps:.2f}s" if fps else ""
    title.set_text(
        f"{path.name}  frame {t}/{len(video)-1}{time_s}\n"
        f"shape (T,H,W)={video.shape}  key={meta.get('key')}"
    )
    fig.canvas.draw_idle()


file_dd = widgets.Dropdown(
    options=[(p.name, p) for p in mat_files],
    description="файл:",
    layout=widgets.Layout(width="420px"),
)
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="кадр:",
    continuous_update=True,
    readout=True,
    layout=widgets.Layout(width="520px"),
)
jump = widgets.BoundedIntText(
    value=0, min=0, max=0, description="перейти:", layout=widgets.Layout(width="180px")
)
btn_go = widgets.Button(description="→", layout=widgets.Layout(width="40px"))


def _set_file(path: Path) -> None:
    print(f"загрузка {path.name}…", flush=True)
    video, meta = load_video(path)
    state.update(video=video, meta=meta, path=path)
    last = len(video) - 1
    frame_slider.max = last
    jump.max = last
    frame_slider.value = 0
    jump.value = 0
    _render_frame(0)
    print(f"готово: T={video.shape[0]} H={video.shape[1]} W={video.shape[2]}")


def _on_file(change) -> None:
    if change["name"] == "value" and change["new"] is not None:
        _set_file(change["new"])


def _on_frame(change) -> None:
    if change["name"] == "value":
        jump.value = change["new"]
        _render_frame(change["new"])


def _on_go(_btn) -> None:
    frame_slider.value = int(jump.value)


file_dd.observe(_on_file, names="value")
frame_slider.observe(_on_frame, names="value")
btn_go.on_click(_on_go)

display(
    widgets.VBox([
        file_dd,
        widgets.HBox([frame_slider, jump, btn_go]),
    ])
)
display(fig)

_set_file(mat_files[0])